In [1]:
#Q11 Valeurs nulles

In [13]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("TradeCorp ETL") \
    .getOrCreate()

PATH = "/home/jovyan/data/"
#  Charger les 8 DataFrames
df_customers = spark.read.csv(PATH + "customers.csv", header=True, inferSchema=True)
df_categories = spark.read.csv(PATH + "categories.csv", header=True, inferSchema=True)
df_employees = spark.read.csv(PATH + "employees.csv", header=True, inferSchema=True)
df_order_details = spark.read.csv(PATH + "order_details.csv", header=True, inferSchema=True)
df_orders = spark.read.csv(PATH + "orders.csv", header=True, inferSchema=True)
df_products = spark.read.csv(PATH + "products.csv", header=True, inferSchema=True)
df_shippers = spark.read.csv(PATH + "shippers.csv", header=True, inferSchema=True)
df_suppliers = spark.read.csv(PATH + "suppliers.csv", header=True, inferSchema=True)

#  regrouper dans un dictionnaire
dfs = {
    "customers": df_customers,
    "categories": df_categories,
    "employees": df_employees,
    "order_details": df_order_details,
    "orders": df_orders,
    "products": df_products,
    "shippers": df_shippers,
    "suppliers": df_suppliers
}
for nom_df, df in dfs.items():
    print(f"\n=== {nom_df.upper()} ===")
    for c in df.columns:
        nb_nulls = df.filter(col(c).isNull()).count()
        print(f"Colonne {c} : {nb_nulls} valeur(s) nulle(s)")


=== CUSTOMERS ===
Colonne customer_id : 0 valeur(s) nulle(s)
Colonne company_name : 0 valeur(s) nulle(s)
Colonne contact_name : 0 valeur(s) nulle(s)
Colonne contact_title : 0 valeur(s) nulle(s)
Colonne address : 0 valeur(s) nulle(s)
Colonne city : 0 valeur(s) nulle(s)
Colonne region : 60 valeur(s) nulle(s)
Colonne postal_code : 1 valeur(s) nulle(s)
Colonne country : 0 valeur(s) nulle(s)
Colonne phone : 0 valeur(s) nulle(s)
Colonne fax : 22 valeur(s) nulle(s)

=== CATEGORIES ===
Colonne category_id : 0 valeur(s) nulle(s)
Colonne category_name : 0 valeur(s) nulle(s)
Colonne description : 0 valeur(s) nulle(s)
Colonne picture : 8 valeur(s) nulle(s)

=== EMPLOYEES ===
Colonne employee_id : 0 valeur(s) nulle(s)
Colonne last_name : 0 valeur(s) nulle(s)
Colonne first_name : 0 valeur(s) nulle(s)
Colonne title : 0 valeur(s) nulle(s)
Colonne title_of_courtesy : 0 valeur(s) nulle(s)
Colonne birth_date : 0 valeur(s) nulle(s)
Colonne hire_date : 0 valeur(s) nulle(s)
Colonne address : 0 valeur(s) nu

In [3]:
#Q12 — Supprimer les nulls

In [4]:
from pyspark.sql.functions import col

# Supprimer les lignes où shipped_date est null dans df_orders
df_orders = df_orders.dropna(subset=["shipped_date"])

# Calculer la médiane de unit_price dans df_products
mediane_price = df_products.approxQuantile("unit_price", [0.5], 0.0)[0]
print(f"Médiane calculée pour unit_price : {mediane_price}")

#  Remplacer les valeurs nulles par la médiane dans df_products
df_products = df_products.fillna({"unit_price": mediane_price})

# --- Vérifications ---
nulls_shipped = df_orders.filter(col("shipped_date").isNull()).count()
nulls_price = df_products.filter(col("unit_price").isNull()).count()

print(f"Nuls restants dans shipped_date (orders) : {nulls_shipped}")
print(f"Nuls restants dans unit_price (products) : {nulls_price}")

Médiane calculée pour unit_price : 19.5
Nuls restants dans shipped_date (orders) : 0
Nuls restants dans unit_price (products) : 0


In [5]:
#Q13 — Cast des types

In [6]:
from pyspark.sql.functions import to_date, col 
from pyspark.sql.types import DoubleType, IntegerType
#  Conversion des trois colonnes de dates avec leur format
df_orders = df_orders \
    .withColumn('order_date', to_date(col('order_date'), 'M/d/yyyy')) \
    .withColumn('required_date', to_date(col('required_date'), 'M/d/yyyy')) \
    .withColumn('shipped_date', to_date(col('shipped_date'), 'M/d/yyyy'))

# Affichage unique du schéma
df_orders.printSchema()

# Conversion des colonnes unit_price et quantity
df_order_details = df_order_details \
    .withColumn("unit_price", col("unit_price").cast(DoubleType())) \
    .withColumn("quantity", col("quantity").cast(IntegerType()))

# Affichage unique du schéma
df_order_details.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- required_date: date (nullable = true)
 |-- shipped_date: date (nullable = true)
 |-- ship_via: integer (nullable = true)
 |-- freight: double (nullable = true)
 |-- ship_name: string (nullable = true)
 |-- ship_address: string (nullable = true)
 |-- ship_city: string (nullable = true)
 |-- ship_region: string (nullable = true)
 |-- ship_postal_code: string (nullable = true)
 |-- ship_country: string (nullable = true)

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount: double (nullable = true)



In [7]:
#Q14 — Nettoyage des chaînes

In [8]:
from pyspark.sql import functions as F

# TRIM sur les colonnes texte
for c in ["company_name", "contact_name", "contact_title",
          "address", "city", "region", "postal_code", "country", "phone", "fax"]:
    df_customers = df_customers.withColumn(c, F.trim(F.col(c)))

#  Mettre contact_name en title
df_customers = df_customers.withColumn("contact_name", F.initcap(F.col("contact_name")))

#  Mettre country en majuscules 
df_customers = df_customers.withColumn("country", F.upper(F.col("country")))

# Vérification du résultat
df_customers.show(5)

+-----------+--------------------+------------------+--------------------+--------------------+-----------+------+-----------+-------+--------------+--------------+
|customer_id|        company_name|      contact_name|       contact_title|             address|       city|region|postal_code|country|         phone|           fax|
+-----------+--------------------+------------------+--------------------+--------------------+-----------+------+-----------+-------+--------------+--------------+
|      ALFKI| Alfreds Futterkiste|      Maria Anders|Sales Representative|       Obere Str. 57|     Berlin|  NULL|      12209|GERMANY|   030-0074321|   030-0076545|
|      ANATR|Ana Trujillo Empa...|      Ana Trujillo|               Owner|Avda. de la Const...|México D.F.|  NULL|      05021| MEXICO|  (5) 555-4729|  (5) 555-3745|
|      ANTON|Antonio Moreno Ta...|    Antonio Moreno|               Owner|     Mataderos  2312|México D.F.|  NULL|      05023| MEXICO|  (5) 555-3932|          NULL|
|      ARO

In [9]:
#Q15 — Renommer les colonnes


In [10]:
#  Renommer les colonnes dans df_order_details
df_order_details = df_order_details.withColumnRenamed("unit_price", "prix_unitaire").withColumnRenamed("quantity", "quantite").show(5)
# Renommer ship_via dans df_orders
df_orders = df_orders.withColumnRenamed("ship_via", "shipper_id").show(5)

+--------+----------+-------------+--------+--------+
|order_id|product_id|prix_unitaire|quantite|discount|
+--------+----------+-------------+--------+--------+
|   10248|        11|         14.0|      12|     0.0|
|   10248|        42|          9.8|      10|     0.0|
|   10248|        72|         34.8|       5|     0.0|
|   10249|        14|         18.6|       9|     0.0|
|   10249|        51|         42.4|      40|     0.0|
+--------+----------+-------------+--------+--------+
only showing top 5 rows
+--------+-----------+-----------+----------+-------------+------------+----------+-------+--------------------+--------------------+--------------+-----------+----------------+------------+
|order_id|customer_id|employee_id|order_date|required_date|shipped_date|shipper_id|freight|           ship_name|        ship_address|     ship_city|ship_region|ship_postal_code|ship_country|
+--------+-----------+-----------+----------+-------------+------------+----------+-------+-----------------

In [11]:
#Q16 — Colonnes calculées

In [14]:
from pyspark.sql.functions import col, round

# Calculer le sous-total 
df_order_details = df_order_details.withColumn(
    "sous_total", 
    round(col("prix_unitaire") * col("quantite") * (1 - col("discount")), 2)
)

# Afficher les résultats
df_order_details.select("prix_unitaire", "quantite", "discount", "sous_total").show(5)

{"ts": "2026-08-31 23:56:15.548", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `prix_unitaire` cannot be resolved. Did you mean one of the following? [`product_id`, `discount`, `unit_price`, `quantity`, `order_id`]. SQLSTATE: 42703", "context": {"file": "line 6 in cell [15]", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o605.withColumn.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `prix_unitaire` cannot be resolved. Did you mean one of the following? [`product_id`, `discount`, `unit_price`, `quantity`, `order_id`]. SQLSTATE: 42703;\n'Project [order_id#1843, product_id#1844, unit_price#1845, quantity#1846, discount#1847, 'round('`*`('`*`('prix_unitaire, 'quantite), (cast(1 as do

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `prix_unitaire` cannot be resolved. Did you mean one of the following? [`product_id`, `discount`, `unit_price`, `quantity`, `order_id`]. SQLSTATE: 42703;
'Project [order_id#1843, product_id#1844, unit_price#1845, quantity#1846, discount#1847, 'round('`*`('`*`('prix_unitaire, 'quantite), (cast(1 as double) - discount#1847)), 2) AS sous_total#3283]
+- Relation [order_id#1843,product_id#1844,unit_price#1845,quantity#1846,discount#1847] csv
